In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from huggingface_hub import InferenceClient, login
from peft import LoraConfig, get_peft_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

login(token="hf_token")

client = InferenceClient(
    model="meta-llama/Llama-4-Scout-17B-16E-Instruct",
    token="hf_token"
)

model_id = "meta-llama/Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
)
model.to(device)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

Using device: cuda


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 120.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_f

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from datasets import load_dataset, concatenate_datasets, ClassLabel, Value, Features

evidence_ds = load_dataset('copenlu/fever_gold_evidence')
health_ds = load_dataset('TheoTsio/Health_Misinfo')

max_samples=3000

def prepare_evidence_ds(example):
    text = example['claim']
    label_str = example['label']
    if label_str == 'SUPPORTS':
        label_int = 1
    elif label_str in ['REFUTES', 'NOT ENOUGH INFO']:
        label_int = 0
    else:
        label_int = 0
    return {'text': text, 'label': label_int}

evidence_reduced = evidence_ds['train'].select(range(min(max_samples,len(evidence_ds['train']))))
evidence_ds_processed = evidence_reduced.map(prepare_evidence_ds, remove_columns=evidence_ds['train'].column_names)

def prepare_health_ds(example):
    text = example['Document']
    label_str = example['Credibility']
    if label_str == 'real_news':
        label_int = 1
    elif label_str == 'fake_news':
        label_int = 0
    else:
        label_int = 0
    return {'text': text, 'label': label_int}

health_reduced = health_ds['train'].select(range(min(max_samples, len(health_ds['train']))))
health_ds_processed = health_reduced.map(prepare_health_ds, remove_columns=health_ds['train'].column_names)

unified_format = concatenate_datasets([
    evidence_ds_processed,
    health_ds_processed,
])

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 96.9MB            

train.jsonl: downloading bytes:           |  0.00B            

valid.jsonl:   0%|          | 0.00/6.47M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/6.63M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/228277 [00:00<?, ? examples/s]

... (10 outputs left out)

Generating validation split:   0%|          | 0/638 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/639 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
amp_device_type = device.type if device.type in ["cuda", "mps"] else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
max_seq_length = 512

def tokenize_function(examples):
    texts = [str(x) for x in examples['text']]
    return tokenizer(texts, truncation=True, padding='max_length', max_length=max_seq_length, return_tensors="pt")

tokenized_dataset = unified_format.map(tokenize_function, batched=True, remove_columns=unified_format.column_names)

def collate_fn(batch):
    input_ids = torch.stack([torch.tensor(item['input_ids']) for item in batch])
    attention_mask = torch.stack([torch.tensor(item['attention_mask']) for item in batch])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": input_ids.clone()}

train_loader = DataLoader(tokenized_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=0.01
)
from contextlib import nullcontext
import torch
import transformers
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
device_type = device.type if device.type in ["cuda", "mps"] else "cpu"
print(f"Using device: {device}")

if hasattr(model, 'get_input_embeddings') and model.get_input_embeddings() is not None:
    model.get_input_embeddings().to(device)

for module in model.modules():
    if isinstance(module, transformers.models.llama.modeling_llama.LlamaRMSNorm):
        module.to(device)

for name, param in model.named_parameters():
    if param.device != device:
        print(f"Moving parameter {name} from {param.device} to {device}")
        param.data = param.data.to(device)
        if param.grad is not None:
            param.grad.data = param.grad.data.to(device)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.train()
epochs = 3
gradient_accumulation_steps = 8



Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Using device: cuda
Moving parameter base_model.model.model.embed_tokens.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.k_proj.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.self_attn.o_proj.weight from cuda:0 to cuda
Moving parameter base_model.model.model.layers.0.mlp.gate_proj.weight from 

In [ ]:
print("Starting native PyTorch training loop...")
for epoch in range(epochs):
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.amp.autocast(device_type=device_type, dtype=torch.bfloat16):
          outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels)
          loss = outputs.loss

        loss = loss / gradient_accumulation_steps

        loss.backward()

        if (step + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

            print(f"Epoch [{epoch+1}/{epochs}], Step [{step+1}/{len(train_loader)}], Loss: {loss.item() * gradient_accumulation_steps:.4f}")

print("Training finished successfully!")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Starting native PyTorch training loop...
Epoch [1/3], Step [8/6000], Loss: 1.8196
Epoch [1/3], Step [16/6000], Loss: 15.3414
Epoch [1/3], Step [24/6000], Loss: 1.7222
Epoch [1/3], Step [32/6000], Loss: 2.0132
Epoch [1/3], Step [40/6000], Loss: 9.8881
Epoch [1/3], Step [48/6000], Loss: 2.2901
Epoch [1/3], Step [56/6000], Loss: 1.7608
Epoch [1/3], Step [64/6000], Loss: 1.8183
Epoch [1/3], Step [72/6000], Loss: 0.8552
Epoch [1/3], Step [80/6000], Loss: 0.9283
Epoch [1/3], Step [88/6000], Loss: 0.5172
Epoch [1/3], Step [96/6000], Loss: 0.3230
Epoch [1/3], Step [104/6000], Loss: 1.9036
Epoch [1/3], Step [112/6000], Loss: 1.7568
Epoch [1/3], Step [120/6000], Loss: 2.6207
Epoch [1/3], Step [128/6000], Loss: 0.1044
Epoch [1/3], Step [136/6000], Loss: 1.8233
Epoch [1/3], Step [144/6000], Loss: 2.5484
Epoch [1/3], Step [152/6000], Loss: 2.9032
Epoch [1/3], Step [160/6000], Loss: 1.8236
Epoch [1/3], Step [168/6000], Loss: 0.1002
Epoch [1/3], Step [176/6000], Loss: 0.0995
Epoch [1/3], Step [184/60

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
torch.save(model.state_dict(), "./fact_check_model_weights.pth")
tokenizer.save_pretrained("./fact_check_trained_model")
print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!


In [ ]:
import os
current_dir = os.getcwd()
print(f"Current Working Directory: {current_dir}")

📂 Current Working Directory: /content


In [ ]:
print("\nFiles/Folders here:")
print(os.listdir(current_dir))


Files/Folders here:
['.config', 'drive', 'fact_check_trained_model', 'fact_check_model_weights.pth', 'sample_data']


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

new_folder_name = "fact_check_trained_model"
drive_save_path = f"/content/drive/MyDrive/{new_folder_name}"

os.makedirs(drive_save_path, exist_ok=True)
print(f"Created new folder: {drive_save_path}")

model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print("\n Success files inside the folder:")
print(os.listdir(drive_save_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Created new folder at: /content/drive/MyDrive/fact_check_trained_model

✅ Successfully saved! Files inside the folder:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
